# KI and Data Science

# Tutorial 4: Linear Regression

- Linear Regression: Data creation, model setup, training

Prerequisites:

- Python

My contact:

- Niklas Beuter (niklas.beuter@th-luebeck.de)

Course:

- Slides and notebooks will be available at https://lernraum.th-luebeck.de/course/view.php?id=5530

<!---[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.google.com/github/)-->
colab

In [ ]:
%pip install pandas requests xlrd matplotlib scikit-learn numpy torch

## Toy dataset

For the initial demonstration of how AI works we use a toy dataset. Former ideas of predicting values or simple classification have been based on "instance" decisions. You just look at examples and you decide due to the closest example(s). The distance between samples could be calculated in different ways. Here, we just use the euclidian distance between the samples in each feature dimension. This is very intuitive as humans tend to do the same in daily live. We come to a lot of decisions based on our experience, where we just compare a situation to known memories and decide based on that.

We start with creating two classes (squares and triangles).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Setting up the data points for two features
triangles = np.array([[2, 5], [3, 4], [3, 6], [4, 5], [5, 4], [6, 6], [7, 5]])
squares = np.array([[3,3],[6, 3], [7, 2], [8, 4], [7, 4], [8, 5], [6, 5]])
new_instance = np.array([4, 3])

# Plot training instances
plt.figure(figsize=(8, 6))
plt.scatter(triangles[:, 0], triangles[:, 1], marker='^', color='gray', s=100, label='Class: Triangles')
plt.scatter(squares[:, 0], squares[:, 1], marker='s', color='lightgray', s=100, label='Class: Squares')

# Plot new instance
plt.scatter(new_instance[0], new_instance[1], color='black', marker='X', s=100, label='New Instance')

# Labels and title
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.title("New Instance")
plt.xlim(0, 10)
plt.ylim(0, 10)
plt.grid(True)
plt.show()

Next, we select the nearest neighbors by hand (this could be done algorithmically, of course). 

In [ ]:
# Plot training instances
plt.figure(figsize=(8, 6))
plt.scatter(triangles[:, 0], triangles[:, 1], marker='^', color='gray', s=100, label='Class: Triangles')
plt.scatter(squares[:, 0], squares[:, 1], marker='s', color='lightgray', s=100, label='Class: Squares')

# Plot new instance
plt.scatter(new_instance[0], new_instance[1], color='black', marker='X', s=100, label='New Instance')

# Nearest neighbors
nearest_neighbors = np.array([[3, 4], [4, 5], [3, 3]])
neighbor_colors = ['green', 'green', 'orange']
markers = ['^', '^', 's']
arrow_scale = 0.8  # Scale factor to shorten arrows

for neighbor, color, marker in zip(nearest_neighbors, neighbor_colors, markers):
    dx = (neighbor[0] - new_instance[0]) * arrow_scale
    dy = (neighbor[1] - new_instance[1]) * arrow_scale
    plt.scatter(neighbor[0], neighbor[1], color=color, marker=marker, s=100)
    plt.arrow(new_instance[0], new_instance[1], dx, dy,
                head_width=0.1, head_length=0.1, fc=color, ec=color)

# Labels and title
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.title("New Instance and Nearest Neighbors")
plt.xlim(0, 10)
plt.ylim(0, 10)
plt.grid(True)
plt.show()

What would be your decision here? As you can see, the closes example is a square, but the two adjacent examples are triangles.

In classic approaches you would form a heuristic, which decision might be better. In machine learning we do not form a decision ourselves, but we use a model to do the decision. The model itself is mostly some kind of mathematical representation like a polynom. But, as the models get more complex we form hyper dimensional polynoms, which would not be possible by hand. But, these models are powerful enough to find very complex mathematical decisions by fitting these hyper dimensional polynoms in training data.

Here, we just use a pseudo function to show that a classic 2 rank polynomial could form a decision boundary.

In [ ]:
# Define a guessed function to separate the classes
def polynomial(x,a,b,c):
    return a * x**2 + b * x + c


# Plot training instances
plt.figure(figsize=(8, 6))
plt.scatter(triangles[:, 0], triangles[:, 1], marker='^', color='green', s=100, label='Class: Triangles')
plt.scatter(squares[:, 0], squares[:, 1], marker='s', color='orange', s=100, label='Class: Squares')

# Plot new instance
plt.scatter(new_instance[0], new_instance[1], color='black', marker='X', s=100, label='New Instance')

# Draw the non-linear (quadratic) decision boundary
points_x = np.array([5, 6, 9])
points_y = np.array([0, 5.5, 4])
# Erstellen der Koeffizientenmatrix für das Gleichungssystem
A = np.vstack([points_x**2, points_x, np.ones(len(points_x))]).T
coefficients = np.linalg.solve(A, points_y)
# Extrahiere die Koeffizienten a, b und c
a, b, c = coefficients
x_values = np.linspace(4, 9, 100)
y_values = polynomial(x_values,a,b,c)
plt.plot(x_values, y_values, 'k--', label='Model')

# Labels and title
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.title("Non-linear Decision Boundary")
plt.xlim(0, 10)
plt.ylim(0, 10)
plt.grid(True)


plt.show()


Every point left, right or top of the function is a triangle and every point below the curve might be a square. You could think now, there are many errors in that decision. But, if you form more complex boundaries you will get a much better decision.

This just has been a toy example to demonstrate the potential use of mathematical models to form such a decison.

# Linear regression example

Linear regression is one of the most basic and widely used statistical and machine learning methods. It is used to model a linear relationship between a dependent variable \(y\) and one or more independent variables \(X\).

## Fundamental Principle

The goal of linear regression is to find a line (or a hyperplane in higher dimensions) that best approximates the data points. The equation for simple linear regression with one independent variable is as follows:

$$ y = w_0x + b + \epsilon $$

- \(y\) is the dependent variable.
- \(x\) is the independent variable.
- \($b$\) is the y-intercept of the regression line.
- \($w_0$\) is the slope of the regression line.
- \($\epsilon$\) represents the error term or residuals, which are the difference between the observed values and the values predicted by the model.

## Areas of Application

Linear regression finds wide application across many fields including economics, engineering, natural sciences, and social sciences to investigate relationships between variables and to make predictions.

## Least Squares Method

The parameters \($w_0$\) and \($b$\) are commonly estimated using the [least squares method](https://en.wikipedia.org/wiki/Least_squares), where the sum of the squared residuals is minimized to achieve the best possible fit to the data.

## Assumptions

For linear regression to be applied, certain assumptions need to be met including linearity, [homoscedasticity](https://www.statisticshowto.com/homoscedasticity/) (equal or similar variances), independence of errors, no or little [multicollinearity](https://www.investopedia.com/terms/m/multicollinearity.asp) (Multicollinearity is the occurrence of high intercorrelations among two or more independent variables in a multiple regression model), and normal distribution of errors.

## Going into the code

First, we need to import some dependencies.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

## Linear Regression example

Next, we are going to implement and use linear regression ourself. First, we create an own dataset. Here, we create a dataset for car price prediction using mileage and age. 

In [ ]:
# First, we create our own dataset. In this case we want to predict the value of cars

# Set seed for reproducibility
np.random.seed(42)

# Generate random data
num_samples = 1000
kilometer = np.random.uniform(0, 200000, num_samples)  # Kilometerstand zwischen 0 und 200.000 km
age = np.random.uniform(0, 15, num_samples)  # Alter zwischen 0 und 15 Jahren

# Define parameters for the linear model
base_price = 30000  # Basispreis eines neuen Fahrzeugs
price_decline_per_km = 0.05  # Preisverfall pro Kilometer
price_decline_per_year = 1500  # Preisverfall pro Jahr

# Calculate vehicle price with some added noise for variability
noise = np.random.normal(0, 1000, num_samples)  # Zufällige Streuung
price = base_price - (kilometer * price_decline_per_km + age * price_decline_per_year) + noise

# Create DataFrame
data = pd.DataFrame({
    'Kilometerstand': kilometer,
    'Alter': age,
    'Preis': price
})

data.head()

In [ ]:
# Quick sneek into the data
data["Preis"][0]

In [ ]:
# plot the created dataset
kilometers = data["Kilometerstand"][:]
age = data["Alter"][:]
prices = data["Preis"][:]

# Plot for kilometers vs price
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.scatter(kilometers, prices, color='blue')
plt.scatter(40000, 18000, color='green')
plt.title('Kilometers vs Price')
plt.xlabel('Kilometers')
plt.ylabel('Price')
plt.grid(True)

# Plot for age vs price
plt.subplot(1, 2, 2)
plt.scatter(age, prices, color='red')
plt.scatter(6, 16000, color='green')
plt.title('Age vs Price')
plt.xlabel('Age')
plt.ylabel('Price')
plt.grid(True)

# Displaying the plots
plt.tight_layout()
plt.show()

### Linear model and loss and Optimization via gradient descent

Our linear regression model is $ y = f(x) = w \cdot x + b$. For an easy start we just use the 'Alter' data for predicting a price. 

We solve for $w$ and $b$ using gradient descent (`SGD = stochastic gradient descent`). We will learn about this later in the course (for now, SGD helps to find optimal parameters in an iterative fashion). We initialize parameters $w$ and $b$ randomly, which is typical for starting a model training in machine learning. The `learning rate` is set to $0.001$ (more about the learning rate will be in the next lectures). The number of epochs are the iterations we use for optimizing our parameters in SGD.

In order to have something to optimize against, we need a mathematical description about the goal we want to achieve. This is called the `Loss function`. Here, we want the model to predict the correct price, which in turn means that the model should predict as close as possible to a known price from the training set. As loss function we use in this case the squared mean between the predicted value $\hat{Y}$ and the real $Y$ value: $L = mean(\hat{Y} - Y)^2$ (also called `MSE = mean squared error`). Again, a loss function is our goal, we want to optimize. Here, we want a prediction as close as possible to the real value (supervised learning). The squared difference helps in neglecting the direction of the difference. The mean is used to get the mean error the system is producing on the training set. Once we have the error we can use it to adapt our parameters to better fit the data.

So, what we are doing here is the following. We create a starting point for our linear regression model by giving the parameters some kind of starting value (here a value between 0 and 1). Then, we iterate a learning cycle, where we pass the data into the model and predict an output. Of course, the output is not fitting to the real value as we guessed the starting point for the model. The difference to the known output from the training data helps us to calculate the loss (MSE). This scalar value is used in the optimizer step to calculate gradients for all parameters, which we use to update all parameters. The trick is to reduce the update step by a factor (the learning rate) and just update all parameters slightly. Doing this many iterations slightly moves the parameters to optimized values (stochastic gradient descent).

In [ ]:
# Convert the DataFrame to PyTorch tensors
# We are not diving into tensors and their operations here, but just use them as a tool to perform our linear regression. 
# For more information, please refer to the PyTorch documentation. https://docs.pytorch.org/tutorials/beginner/introyt/tensors_deeper_tutorial.html
# torch is a popular deep learning library called PyTorch that provides tools for tensor computations and automatic differentiation.
X = torch.tensor(data['Alter'].values, dtype=torch.float32).view(-1, 1)  # Reshaping for matrix operations
Y = torch.tensor(data['Preis'].values, dtype=torch.float32).view(-1, 1)

# Initialize model parameters w and b (here, we define our model ourselves)
w = torch.randn(1, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

# Define the optimizer (SGD = stochastic gradient descent, lr is the learning rate. It defines the speed of update for each learning iteration)
optimizer = torch.optim.SGD([w, b], lr=0.001)

# Training loop
num_epochs = 10000
for epoch in range(num_epochs):
    # Forward pass: Compute predicted Y by passing X to the model. We use matrix multiplication here for speed reasons.
    # Linear model equation: Y_pred = X * w + b (@ overloaded matrix multiplication)
    Y_pred = X @ w + b

    # Compute loss: Mean Squared Error (MSE)
    loss = torch.mean((Y_pred - Y) ** 2)

    # Zero gradients -> Is always done before each iteration to reset all gradient calculations from the last step  
    optimizer.zero_grad()
    # perform a backward pass -> computes the gradient of the loss for every parameter that has requires_grad=True.
    loss.backward()
    # and update the weights -> updates the parameters based on the current gradients.
    optimizer.step()

    # Print loss every 1000 epochs
    if epoch % 1000 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')

# Print final parameters
print(f'w: {w.item()}, b: {b.item()}')

Have a look at the loss value between each iteration. This value should decrease as the difference between the predictions and real values should get less. Observing this value is crucial for deep learning as we can see, if our parameter selection (initialization, learning rate) is good and leads to a good model. We will come back to this, what it means, if this value is not decreasing or even raising.

At the end, you should always do a visualization of your result, here the linear fit.

In [ ]:
# Visualization
plt.figure(figsize=(10, 6))
plt.scatter(X.numpy(), Y.numpy(), color='blue', label='Actual data', alpha=0.5)
plt.plot(X.numpy(), (X @ w + b).detach().numpy(), color='red', label='Regression Line')
plt.xlabel('Alter')
plt.ylabel('Preis')
plt.title('Linear Regression Model')
plt.legend()
plt.show()

### Connection of Loss and Optimizer

You might have wondered how the optimizer knows about the loss and the gradients. The cool thing is the optimizer in PyTorch doesn't need to be explicitly told about the loss during its initialization. Instead, it works through the connection established by the computational graph that PyTorch builds during the forward pass and the subsequent .backward() call on the loss tensor.

In machine / deep learning you are always doing the following steps, which establish the connection between loss and optimizer:

* **Forward Pass**: You compute the predicted values (Y_pred) based on your model's parameters (w and b in this case) and inputs (X). This operation, along with the loss computation that follows, constructs a computational graph. This graph contains information about which operations were performed on the tensors and how they are connected.

* **Compute Loss**: The loss is computed using the predictions and the actual values (Y). This computation also becomes part of the computational graph. The loss tensor directly or indirectly references the model parameters that contributed to its value.

* **Backward Pass**: When loss.backward() is called, PyTorch traverses this graph starting from the loss tensor backward to all tensors that have requires_grad=True set and computes the gradient of the loss with respect to these tensors. This is where the connection between the loss and the optimizer's parameters is made use of. The gradients are stored in the .grad attribute of each tensor (parameter) involved in the computation of the loss.

* **Optimizer Step**: The optimizer is initialized with a list of parameters ([w, b] in this example). After the backward pass, each of these parameters has its .grad attribute populated with the gradients computed by .backward(). When optimizer.step() is called, it updates each parameter by taking a step in the direction that minimally decreases the loss, using the gradients stored in .grad. This step is determined by the optimization algorithm the optimizer implements (SGD in this case) and the learning rate.

### Prediction of new values

Now we can start predicting a price for other cars using the trained model. Play with different values.

In [ ]:
# New data for prediction
new_alter = torch.tensor([10.0])  # For example, predicting price for a car of age 10 years

# Predicting the price using the trained model
predicted_price = new_alter * w.detach() + b.detach()

print(f'Predicted Price: {predicted_price.item()}€')

## Multiple linear regression

In multiple linear regression, each input feature has its own coefficient (weight), and the model learns the relationship between each feature and the target variable.

The linear model equation for multiple linear regression, considering both "Alter" and "Kilometerstand" as input features, would be:

$ Preis = w_0 * Alter + w_1 * Kilometerstand + b $

As the input data is not equally scaled (Kilometer vs Age) the model would not learn a good prediction. We need to bring it to a similar value range. This is done by `normalization` using a subtraction of the mean and a division by the standard deviation. First, we try without normalization to see the effect. Then, we apply normalization and see the difference in learning performance.

see: [Normalization - Wikipedia](https://en.wikipedia.org/wiki/Normalization_(statistics))

In [ ]:
# Assuming 'data' is your DataFrame
# Convert the DataFrame to PyTorch tensors. This time, include both 'Alter' and 'Kilometerstand'.
X = torch.tensor(data[['Alter', 'Kilometerstand']].values, dtype=torch.float32)
Y = torch.tensor(data['Preis'].values, dtype=torch.float32).view(-1, 1)

# Normalize features
# X_norm = (X - X.mean(dim=0)) / X.std(dim=0)
X_norm = X

# Initialize model parameters w for both features and b
w = torch.randn(2, 1, requires_grad=True)  # Now w has two weights: one for 'Alter' and one for 'Kilometerstand'
b = torch.randn(1, requires_grad=True)

# Define the optimizer
optimizer = torch.optim.SGD([w, b], lr=1e-4)  # Adjust the learning rate if necessary

# Training loop
num_epochs = 25000
for epoch in range(num_epochs):
    # Forward pass: Compute predicted Y by passing X to the model
    Y_pred = X_norm @ w + b  # Matrix multiplication to handle multiple features

    # Compute loss: Mean Squared Error (MSE)
    loss = torch.mean((Y_pred - Y) ** 2)

    # Zero gradients, perform a backward pass, and update the weights.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print loss every 1000 epochs
    if epoch % 1000 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')

# Print final parameters
print(f'w: {w.detach().view(-1).numpy()}, b: {b.detach().item()}')

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # Import the 3D plotting module

# Assuming w and b are your model's parameters after training
# And X and Y are your feature matrix and target vector

# Prepare data for plotting
X_plot = X_norm[:, 0].numpy()  # Alter
Y_plot = X_norm[:, 1].numpy()  # Kilometerstand
Z_plot = Y.numpy()        # Preis (actual prices)

# Predicted prices for plotting the regression plane
Z_pred = (X_norm @ w + b).detach().numpy()

# Creating a 3D plot
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Plot actual data points
ax.scatter(X_plot, Y_plot, Z_plot, color='b', label='Actual data')

# Create a mesh grid for the plane
X_grid, Y_grid = np.meshgrid(np.linspace(X_plot.min(), X_plot.max(), 20),
                             np.linspace(Y_plot.min(), Y_plot.max(), 20))

# Predict Z values (prices) on the mesh grid
Z_grid = (w[0].item() * X_grid + w[1].item() * Y_grid + b.item())

# Plot the regression plane
ax.plot_surface(X_grid, Y_grid, Z_grid, color='r', alpha=0.5)

ax.set_xlabel('Alter')
ax.set_ylabel('Kilometerstand')
ax.set_zlabel('Preis')
ax.legend()

plt.show()

In [ ]:
# Assuming 'X_mean' and 'X_std' are the mean and standard deviation of the training data, used for normalization

# New data for prediction (e.g., 'Alter' and 'Kilometerstand')
new_data = torch.tensor([[10.0, 150000.0]])  # For example, a 10-year-old car with 50,000 km

# Normalize new data if the model was trained with normalized features
new_data_norm = (new_data - X.mean(dim=0)) / X.std(dim=0)

# Predicting the price using the trained model
predicted_price = new_data_norm @ w.detach() + b.detach()

print(f'Predicted Price: {predicted_price.item()}€')

# Happiness index

As in the course lecture we want to predict the happiness of a country by its GPD.

In [ ]:
import pandas as pd
import requests
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

# Step 1: Download GDP per capita data from World Bank API
gdp_per_capita_url = "http://api.worldbank.org/v2/en/indicator/NY.GDP.PCAP.CD?downloadformat=excel"

# Download the GDP per capita data
response = requests.get(gdp_per_capita_url)
with open("world_bank_gdp_per_capita_data.xls", "wb") as file:
    file.write(response.content)

# Load the GDP per capita data into a DataFrame
gdp_per_capita_data = pd.read_excel("world_bank_gdp_per_capita_data.xls", sheet_name="Data", skiprows=3)

# Clean and preprocess GDP per capita data
gdp_per_capita_data.dropna(axis=1, how='all', inplace=True)
gdp_per_capita_data.rename(columns={"Country Name": "Country", "Country Code": "Country_Code"}, inplace=True)
gdp_per_capita_data.set_index("Country", inplace=True)
latest_year = gdp_per_capita_data.columns[-1]
gdp_per_capita_latest = gdp_per_capita_data[[latest_year]].dropna()
gdp_per_capita_latest.rename(columns={latest_year: "GDP_per_Capita"}, inplace=True)

# Step 2: Load OECD Happiness Data (adapt the path to your own path)
oecd_data = pd.read_csv("../presentations/data/OECD_happyness.csv")

# Extract relevant columns from OECD data and rename for clarity
oecd_data_filtered = oecd_data[['Country', 'OBS_VALUE']].rename(columns={"OBS_VALUE": "Happiness_Index"})

# Step 3: Merge GDP per capita data with OECD Happiness Data
combined_data = pd.merge(oecd_data_filtered, gdp_per_capita_latest, on="Country", how="inner")

# Step 4: Drop rows with any missing values and double values
combined_data.dropna(inplace=True)
combined_data.drop_duplicates(subset="Country", keep="first", inplace=True)
# We also drop the lowest and the two highest GDP income countries
combined_data_drop = combined_data.drop(combined_data.nsmallest(1, 'GDP_per_Capita').index)
combined_data_drop = combined_data_drop.drop(combined_data_drop.nlargest(2, 'GDP_per_Capita').index)

# Step 5: Plot Happiness Index vs GDP per Capita
plt.figure(figsize=(10, 6))
plt.scatter(combined_data_drop["GDP_per_Capita"], combined_data_drop["Happiness_Index"], alpha=0.7)
plt.title("Happiness Index vs GDP per Capita by Country")
plt.xlabel("GDP per Capita")
plt.ylabel("Happiness Index")
plt.grid(True)

# Display the plot
plt.show()

In [ ]:
# Juste execute, in case you want to save the result data file
#combined_data.sort_values(by="GDP_per_Capita", ascending=False, inplace=True)
#combined_data.to_csv("../presentations/data/gdp_by_country.csv")
print(combined_data)

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Step 6: Prepare data for linear regression
X = combined_data_drop["GDP_per_Capita"].values.reshape(-1, 1)
y = combined_data_drop["Happiness_Index"].values

# Fit linear regression model
model = LinearRegression()
model.fit(X, y)

# Predict values for the regression line
x_range = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
y_pred = model.predict(x_range)

# Step 7: Prepare data for linear regression
X_orig = combined_data["GDP_per_Capita"].values.reshape(-1, 1)
y_orig = combined_data["Happiness_Index"].values

# Fit linear regression model
model_orig = LinearRegression()
model_orig.fit(X_orig, y_orig)

# Predict values for the regression line
x_range_orig = np.linspace(X_orig.min(), X_orig.max(), 100).reshape(-1, 1)
y_pred_orig = model_orig.predict(x_range_orig)

lowest_gdp = combined_data.nsmallest(1, "GDP_per_Capita")
highest_gdp = combined_data.nlargest(2, "GDP_per_Capita")

# Step 8: Plot Happiness Index vs GDP per Capita with the regression line
plt.figure(figsize=(10, 6))
plt.scatter(combined_data_drop["GDP_per_Capita"], combined_data_drop["Happiness_Index"], alpha=0.7, label="Data")
plt.scatter(lowest_gdp["GDP_per_Capita"], lowest_gdp["Happiness_Index"], color="blue", marker="v", s=100, label="Lowest GDP")
plt.scatter(highest_gdp["GDP_per_Capita"], highest_gdp["Happiness_Index"], color="purple", marker="^", s=100, label="Highest GDP")
plt.plot(x_range, y_pred, color="red", label="Linear Regression Line")
plt.plot(x_range_orig, y_pred_orig, color="green", label="Linear Regression Line All")
plt.title("Happiness Index vs GDP per Capita by Country with Regression Line")
plt.xlabel("GDP per Capita")
plt.ylabel("Happiness Index")
plt.legend()
plt.grid(True)

# Display the plot
plt.show()

## Gradient descent visualization

Next, we use a stochastic gradient descent regressor (SGDRegresser, kind of linear regression) to show the single steps of the parameters during training. First, we scale the data as the SGD regressor performs better with scaled data. Then, we plot every training step as a regression curve to demonstrate how SGD improves step by step to a good decision.

In [ ]:


# Step 1: Prepare the data
X = combined_data_drop["GDP_per_Capita"].values.reshape(-1, 1)
y = combined_data_drop["Happiness_Index"].values.reshape(-1, 1)

# Standardize X and y separately for SGD to work correctly
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# Step 2: Initialize the SGD Regressor with a small number of iterations for better visualization
sgd_regressor = SGDRegressor(max_iter=1, warm_start=True, penalty=None, learning_rate="constant", eta0=0.01)

# Step 3: Train in multiple steps, capturing each intermediate line
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.7, label="Data")

# Plot each intermediate line from SGD training
for epoch in range(1, 101):  # 100 iterations for visualization
    sgd_regressor.fit(X_scaled, y_scaled.ravel())  # Train with one additional epoch
    
    # Get the line parameters (coef and intercept) at this training step
    coef = sgd_regressor.coef_[0]
    intercept = sgd_regressor.intercept_
    
    # Generate line points in the scaled space and transform back to original scale
    line_x = np.linspace(X_scaled.min(), X_scaled.max(), 100).reshape(-1, 1)
    line_y = line_x * coef + intercept  # Linear equation in scaled space
    line_x_orig = scaler_X.inverse_transform(line_x)  # Transform X back to original scale
    line_y_orig = scaler_y.inverse_transform(line_y.reshape(-1, 1))  # Transform y back to original scale

    # Plot the intermediate line with a faded color
    plt.plot(line_x_orig, line_y_orig, color="gray", alpha=0.1)

# Final regression line in red
final_y_orig = scaler_y.inverse_transform((X_scaled * coef + intercept).reshape(-1, 1))
plt.plot(X, final_y_orig, color="red", label="Final Regression Line")

# Labels and grid
plt.xlabel("GDP per Capita")
plt.ylabel("Happiness Index")
plt.legend()
plt.grid(True)
plt.title("All Intermediate Regression Lines During Training with SGD")

# Display the plot
plt.show()

## Housing Dataset

We take the housing data set from the lecture again. [**StatLib Household**](https://www.kaggle.com/datasets/harrywang/housing).
1. you have already performed the data cleansing last time.
2. now you have to divide the data set into suitable sub-sets so that you can train with it.
3. train a linear regression in order to be able to predict house prices

In [ ]:
%pip install pandas numpy matplotlib scikit-learn pillow

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Load the dataset (adapt to your path)
file_path = '../presentations/data/california_housing/housing.csv'
df = pd.read_csv(file_path, delimiter=",")  # Assuming delimiter is comma

# Sometimes better than removing corrupt lines is filling with median values
imputer = SimpleImputer(strategy="median")
# we drop ocean proximity, as we can't compute a median of a string (only dropped in the copy)
df_numeric = df.drop("ocean_proximity", axis=1)
imputer.fit(df_numeric)
X = imputer.transform(df_numeric)
df_clean = pd.DataFrame(X, columns=df_numeric.columns, index=df_numeric.index)

# In order to have a better representation for the columns, we adapt the district numbers to single house numbers
# calculate rooms per house, not per district
df_clean.loc[:, "rooms_per_household"] = df_clean["total_rooms"] / df_clean["households"]
df_clean.loc[:, "bedrooms_per_household"] = df_clean["total_bedrooms"] / df_clean["total_rooms"]
df_clean.loc[:, "population_per_household"] = df_clean["population"] / df_clean["households"]

In [ ]:
print(df_clean.head())

### Visualization of features

If you want to analyze features and their correlations you could do it again here. But, as we did the data analysis already last time, we will focus on multi-feature regression.

In [ ]:
import matplotlib.pyplot as plt

# Create scatter-plot
plt.figure(figsize=(10, 6))
plt.scatter(df_clean["median_income"], df_clean["median_house_value"], alpha=0.5)
plt.title("Scatter Plot of Median Income vs Median House Value")
plt.xlabel("Median Income")
plt.ylabel("Median House Value")
plt.show()

In [ ]:
# scale data normalization
attributes = list(df_clean)

# data standardization
scaler = StandardScaler()
df_clean[attributes] = scaler.fit_transform(df_clean[attributes])

The next function is an example of how a train and test split could work. As we can use frameworks for that we focus on the function from sklearn.

In [ ]:
# Create a test and training data split (alternative: use sklearn function train_test_split())
def split_train_test(data, test_ratio, random_state):
    np.random.seed(random_state)
    shuffled_indices = np.random.permutation(len(data))
    test_set_size = int(len(data) * test_ratio)
    test_indices = shuffled_indices[:test_set_size]
    train_indices = shuffled_indices[test_set_size:]
    return data.iloc[train_indices], data.iloc[test_indices]

In [ ]:
# split test and train data (you could use our own function or sklearn function, here sklearn is used)
train_set, test_set = train_test_split(df_clean, test_size=0.2, random_state=42)

# create a label dataset (our training goal) using our house price values
labels = train_set["median_house_value"].copy()
# we drop median house value from the train set as we want to predict this value
train = train_set.drop("median_house_value", axis=1)

# Train linear Regression using sklearn class LinearRegression. The "fit" function trains the model with the data.
lin_reg = LinearRegression()
lin_reg.fit(train, labels)

In [ ]:
# we drop median house value from the test set as we want to predict this value
# we pick some random data from the test set
some_data = test_set.drop("median_house_value", axis=1).iloc[5:10]
some_labels = train_set["median_house_value"].copy().iloc[5:10]

# make predictions using the linear regression model
predictions_df = lin_reg.predict(some_data)
print("Predictions:", predictions_df)
print("Labels:", list(some_labels))

# let us scale back the values as we scaled them initially
idx = 8 # id of median_house_value
scales = scaler.scale_[idx]
mean = scaler.mean_[idx]
predictions_df = predictions_df * scales + mean
label_df = some_labels * scales + mean
print("Predictions scaled:", predictions_df)
print("Labels scaled:", list(label_df))

### What is your opinion concerning the results?

1. Write down your thoughts. Could you imagine, why the results are like that?

### Calculate error done by model

As error criteria we use again the mean squared error. We calculate the full error at the end of the training on the complete training dataset.

In [ ]:
from sklearn.metrics import mean_squared_error

price_predictions_training = lin_reg.predict(train)
lin_mse = mean_squared_error(labels, price_predictions_training)
lin_rmse = np.sqrt(lin_mse)
lin_rmse

# References

This notebook is adapted from or uses following sources:
* Markus Enzweiler, markus.enzweiler@hs-esslingen.de, https://github.com/menzHSE/cv-ml-lecture-notebooks.git
* Aurélien Géron, Praxiseinstieg Machine Learning, 2020